# Advanced Feature Engineering

This notebook creates advanced features for modeling:
- Game situation features (even strength, power play, etc.)
- Time-based features (period, time remaining, etc.)
- Shot quality metrics (danger zones, quality scores)
- Rebound features
- Shot type encoding
- Zone encoding


In [1]:
import pandas as pd
import numpy as np
import sys

# Add src to path
sys.path.append('../../')
from src.features.feature_engineering import extract_game_situation, time_to_seconds


## Load Data with Basic Features


In [2]:
# Load data with basic features (distance, angle)
df_shots_clean = pd.read_parquet("../../data/processed/shots_with_basic_features.parquet")

print(f"Data shape: {df_shots_clean.shape}")
df_shots_clean.head()


Data shape: (680581, 34)


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,yCoord,zoneCode,period,periodType,is_goal,distance,angle,distance_bin,angle_abs,angle_bin
0,10,00:25,19:35,1551,right,505,goal,11,8477015.0,1.0,...,-1.0,O,1,REG,1,4.123106,-14.036243,0-10,14.036243,0-15°
1,12,00:38,19:22,1551,right,507,missed-shot,15,None,None,...,-37.0,O,1,REG,0,71.344236,-31.239215,70-80,31.239215,30-45°
2,15,01:31,18:29,1451,right,506,shot-on-goal,28,None,None,...,2.0,O,1,REG,0,57.035077,2.009554,50-60,2.009554,0-15°
3,50,01:39,18:21,1451,right,508,blocked-shot,32,None,None,...,-7.0,D,1,REG,0,16.552945,-25.016893,10-20,25.016893,15-30°
4,18,01:58,18:02,1451,right,507,missed-shot,33,None,None,...,16.0,O,1,REG,0,45.880279,20.409883,40-50,20.409883,15-30°


## Game Situation Features


In [3]:
print("Creating game situation features...")

# Apply game situation extraction
df_shots_clean['game_situation'] = df_shots_clean['situationCode'].apply(extract_game_situation)

# Create binary features
df_shots_clean['is_even_strength'] = (df_shots_clean['game_situation'] == 'even_strength').astype(int)
df_shots_clean['is_empty_net'] = (df_shots_clean['game_situation'] == 'empty_net').astype(int)
df_shots_clean['is_man_advantage'] = (df_shots_clean['game_situation'] == 'man_advantage').astype(int)

print("\nGame Situation Distribution:")
print(df_shots_clean['game_situation'].value_counts())
print("\nGoal Rate by Game Situation:")
print(df_shots_clean.groupby('game_situation')['is_goal'].agg(['mean', 'count']))


Creating game situation features...

Game Situation Distribution:
game_situation
even_strength    663376
man_advantage     13946
empty_net          3259
Name: count, dtype: int64

Goal Rate by Game Situation:
                    mean   count
game_situation                  
empty_net       0.316048    3259
even_strength   0.051301  663376
man_advantage   0.134662   13946


## Time-based Features


In [4]:
print("\nCreating time-based features...")

# Convert time strings to seconds
df_shots_clean['time_in_period_seconds'] = df_shots_clean['timeInPeriod'].apply(time_to_seconds)
df_shots_clean['time_remaining_seconds'] = df_shots_clean['timeRemaining'].apply(time_to_seconds)

# Time remaining in game (approximate - 20 min periods)
df_shots_clean['time_remaining_game'] = (
    (df_shots_clean['period'] - 1) * 1200 + df_shots_clean['time_remaining_seconds']
)

# Period type features
df_shots_clean['is_regulation'] = (df_shots_clean['periodType'] == 'REG').astype(int)
df_shots_clean['is_overtime'] = (df_shots_clean['periodType'] == 'OT').astype(int)
df_shots_clean['is_shootout'] = (df_shots_clean['periodType'] == 'SO').astype(int)

# Late game indicator (last 5 mins of regulation)
df_shots_clean['is_late_game'] = (
    (df_shots_clean['period'] <= 3) &
    (df_shots_clean['time_remaining_seconds'] <= 300)
).astype(int)

# Early period indicator (first 2 mins)
df_shots_clean['is_early_period'] = (
    df_shots_clean['time_remaining_seconds'] >= 1080
).astype(int)

print("Time-based features created!")
print("\nGoal Rate by Period Type:")
print(df_shots_clean.groupby('periodType')['is_goal'].agg(['mean', 'count']))



Creating time-based features...
Time-based features created!

Goal Rate by Period Type:
                mean   count
periodType                  
OT          0.106458    8238
REG         0.052424  669275
SO          0.318449    3068


## Shot Quality Metrics


In [5]:
print("\nCreating Shot Quality Metrics...")

# Distance-angle interaction
df_shots_clean['distance_angle_interaction'] = (
    df_shots_clean['distance'] * df_shots_clean['angle'].abs()
)

# Normalize distance and angle
df_shots_clean['normalized_distance'] = df_shots_clean['distance'] / 100
df_shots_clean['normalized_angle'] = df_shots_clean['angle'].abs() / 90

# Combined quality metric (lower = better shot)
df_shots_clean['shot_quality_score'] = (
    df_shots_clean['normalized_distance'] * 0.6 +
    df_shots_clean['normalized_angle'] * 0.4
)

# High danger area (slot area = close and centered)
df_shots_clean['is_high_danger'] = (
    (df_shots_clean['distance'] < 25) &
    (df_shots_clean['angle'].abs() < 30)
).astype(int)

# Medium danger area
df_shots_clean['is_medium_danger'] = (
    (df_shots_clean['distance'] < 40) &
    (df_shots_clean['angle'].abs() < 45) &
    ~df_shots_clean['is_high_danger']
).astype(int)

# Low danger area (everything else)
df_shots_clean['is_low_danger'] = (
    ~df_shots_clean['is_high_danger'] &
    ~df_shots_clean['is_medium_danger']
).astype(int)

print("Shot quality metrics created!")
print("\nGoal Rate by Danger Zone:")
danger_zones = []
for zone in ['is_high_danger', 'is_medium_danger', 'is_low_danger']:
    goal_rate = df_shots_clean[df_shots_clean[zone] == 1]['is_goal'].mean()
    count = df_shots_clean[df_shots_clean[zone] == 1][zone].sum()
    danger_zones.append({
        'zone': zone.replace('is_', '').replace('_', ' ').title(),
        'goal_rate': goal_rate,
        'shot_count': count
    })
print(pd.DataFrame(danger_zones))



Creating Shot Quality Metrics...
Shot quality metrics created!

Goal Rate by Danger Zone:
            zone  goal_rate  shot_count
0    High Danger   0.089740      155961
1  Medium Danger   0.057923      184330
2     Low Danger        NaN           0


## Rebound Features


In [6]:
print("\nCreating rebound features...")

# Sort by eventId to maintain chronological order
df_shots_clean_sorted = df_shots_clean.sort_values('eventId').reset_index(drop=True)

# Create shifted version to compare with previous shot
df_shots_clean_sorted['prev_event_id'] = df_shots_clean_sorted['eventId'].shift(1)
df_shots_clean_sorted['event_id_diff'] = (
    df_shots_clean_sorted['eventId'] - df_shots_clean_sorted['prev_event_id']
)

# Consider it a potential rebound if events are close together
df_shots_clean_sorted['is_potential_rebound'] = (
    (df_shots_clean_sorted['event_id_diff'] <= 5) &
    (df_shots_clean_sorted['event_id_diff'] > 0)
).astype(int)

# Update original dataframe
df_shots_clean['is_potential_rebound'] = df_shots_clean_sorted['is_potential_rebound'].values

print("Rebound features created!")
print("\nGoal Rate for Potential Rebounds:")
print(df_shots_clean.groupby('is_potential_rebound')['is_goal'].agg(['mean', 'count']))



Creating rebound features...
Rebound features created!

Goal Rate for Potential Rebounds:
                          mean   count
is_potential_rebound                  
0                     0.054291  678768
1                     0.049090    1813


## Shot Type Features


In [7]:
print("\nCreating shot type features...")

# Create binary features for common shot types
common_shot_types = ['wrist', 'snap', 'slap', 'tip-in', 'backhand', 'deflected']
for shot_type in common_shot_types:
    df_shots_clean[f'is_{shot_type}'] = (
        df_shots_clean['shotType'] == shot_type
    ).astype(int)

print("Shot type features created!")
print("\nGoal Rate by Shot Type (Top 6):")
top_shot_types = df_shots_clean.groupby('shotType')['is_goal'].agg(['mean', 'count'])
top_shot_types = top_shot_types.sort_values('count', ascending=False).head(6)
print(top_shot_types)



Creating shot type features...
Shot type features created!

Goal Rate by Shot Type (Top 6):
              mean   count
shotType                  
wrist     0.068017  278415
unknown   0.000287  170897
snap      0.088081   69777
slap      0.049323   67839
tip-in    0.091698   38485
backhand  0.091311   37531


## Zone Features


In [8]:
print("\nCreating zone features...")

# One-hot encode zones
df_shots_clean['is_offensive_zone'] = (df_shots_clean['zoneCode'] == 'O').astype(int)
df_shots_clean['is_defensive_zone'] = (df_shots_clean['zoneCode'] == 'D').astype(int)
df_shots_clean['is_neutral_zone'] = (df_shots_clean['zoneCode'] == 'N').astype(int)

print("Zone features created!")
print("\nGoal Rate by Zone:")
print(df_shots_clean.groupby('zoneCode')['is_goal'].agg(['mean', 'count']))



Creating zone features...
Zone features created!

Goal Rate by Zone:
              mean   count
zoneCode                  
D         0.002697  174278
N         0.029777   13702
None      0.000000       1
O         0.073207  492600


## Save Final Feature-Engineered Data


In [9]:
# Save final feature-engineered data
OUTPUT_FILE = "../../data/processed/shots_with_all_features.parquet"
df_shots_clean.to_parquet(OUTPUT_FILE, index=False)

print(f"Feature-engineered data saved to {OUTPUT_FILE}")
print(f"\nFinal DataFrame shape: {df_shots_clean.shape}")
print(f"\nTotal features: {len(df_shots_clean.columns)}")
print(f"\nNew features created:")
new_features = [col for col in df_shots_clean.columns if col not in 
                ['eventId', 'timeInPeriod', 'timeRemaining', 'situationCode', 
                 'typeDescKey', 'typeCode', 'sortOrder', 'assist1PlayerId', 
                 'assist1PlayerTotal', 'assist2PlayerId', 'assist2PlayerTotal',
                 'awaySOG', 'awayScore', 'blockingPlayerId', 'eventOwnerTeamId',
                 'goalieInNetId', 'homeSOG', 'homeScore', 'scoringPlayerId',
                 'scoringPlayerTotal', 'shootingPlayerId', 'xCoord', 'yCoord',
                 'zoneCode', 'period', 'periodType', 'shotType', 'homeTeamDefendingSide']]
print(f"  {len(new_features)} engineered features")


Feature-engineered data saved to ../../data/processed/shots_with_all_features.parquet

Final DataFrame shape: (680581, 63)

Total features: 63

New features created:
  35 engineered features
